# 7-day change bins — logistic regression

We do **not** predict the exact case count. We predict **which size of move** happens by day t+7, relative to today.

**Toy story.** Today a ZIP has 12 cases. In 7 days it has 24. Relative change = (24−12)/12 = +100%. That lands in `up_100p`.  
If today had only 2 cases and day t+7 had 6, raw % is also +200%, but base &lt; 10 so we force **`flat`** — we refuse to panic on tiny counts.

Plan file: `CHANGE_BIN_7D_PLAN.md`. Phases below: labels → fit → evaluate → save.

## Phase 1 — Paths, rules, features, labels

### Cell goal
Load the supervised ZIP-day panel, write the labeling rules to disk, build slope predictors, build the 7-day change label, and print how often each bin appears.

### Abstract example
Imagine three days of cases: 5, 8, 11.  
A 3-day slope ending today is (11−5)/2 = 3 cases/day.  
Velocity is today’s change vs yesterday: 11−8 = 3.  
We keep slope **magnitude** (not angle) because larger numbers mean steeper climbs.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

HERE = Path.cwd()
if HERE.name != "Dhruvesh":
    HERE = HERE / "Dhruvesh"
assert HERE.is_dir(), "Open from repo root or Dhruvesh."

candidates = [
    HERE.parent / "Weikun" / "supervised_horizon_outputs" / "supervised_horizon_1_next_day.csv.gz",
    HERE / "Weikun" / "supervised_horizon_outputs" / "supervised_horizon_1_next_day.csv.gz",
    Path.cwd() / "Weikun" / "supervised_horizon_outputs" / "supervised_horizon_1_next_day.csv.gz",
]
DATA = next(p for p in candidates if p.is_file())
OUTPUT = HERE / "change_bin_7d_outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)

BASE_MIN = 10
HORIZON = 7

LABEL_ORDER = [
    "down_300p", "down_100p", "down_50p", "down_10p",
    "flat",
    "up_50p", "up_100p", "up_300p", "up_300p_plus",
]

LABEL_RULES = {
    "horizon_days": HORIZON,
    "relative_change": "(cases_tplus7 - cases_today) / cases_today",
    "softening": f"if cases_today < {BASE_MIN} (or 0), label = flat",
    "bins_when_base_ge_min": {
        "down_300p": "r <= -3.0",
        "down_100p": "-3.0 < r <= -1.0",
        "down_50p": "-1.0 < r <= -0.5",
        "down_10p": "-0.5 < r < -0.1",
        "flat": "-0.1 <= r <= 0.1",
        "up_50p": "0.1 < r <= 0.5",
        "up_100p": "0.5 < r <= 1.0",
        "up_300p": "1.0 < r <= 3.0",
        "up_300p_plus": "r > 3.0",
    },
    "predictors": [
        "case_velocity", "case_acceleration", "case_jerk",
        "slope_3d", "slope_3d_lag1", "slope_3d_lag2",
        "avg_slope_10d",
        "population", "population_density_per_sq_km",
    ],
}

with open(OUTPUT / "label_rules.json", "w") as f:
    json.dump(LABEL_RULES, f, indent=2)

print("Data:", DATA.resolve())
print("Output:", OUTPUT.resolve())
print("Saved labeling rules ->", OUTPUT / "label_rules.json")
print(json.dumps(LABEL_RULES, indent=2))

Data: /Users/dhruvesh/Desktop/Fall 2026/Analytics/Jupyter/Weikun/supervised_horizon_outputs/supervised_horizon_1_next_day.csv.gz
Output: /Users/dhruvesh/Desktop/Fall 2026/Analytics/Jupyter/Dhruvesh/change_bin_7d_outputs
Saved labeling rules -> /Users/dhruvesh/Desktop/Fall 2026/Analytics/Jupyter/Dhruvesh/change_bin_7d_outputs/label_rules.json
{
  "horizon_days": 7,
  "relative_change": "(cases_tplus7 - cases_today) / cases_today",
  "softening": "if cases_today < 10 (or 0), label = flat",
  "bins_when_base_ge_min": {
    "down_300p": "r <= -3.0",
    "down_100p": "-3.0 < r <= -1.0",
    "down_50p": "-1.0 < r <= -0.5",
    "down_10p": "-0.5 < r < -0.1",
    "flat": "-0.1 <= r <= 0.1",
    "up_50p": "0.1 < r <= 0.5",
    "up_100p": "0.5 < r <= 1.0",
    "up_300p": "1.0 < r <= 3.0",
    "up_300p_plus": "r > 3.0"
  },
  "predictors": [
    "case_velocity",
    "case_acceleration",
    "case_jerk",
    "slope_3d",
    "slope_3d_lag1",
    "slope_3d_lag2",
    "avg_slope_10d",
    "population

### Build the modeling table

What happens next:

1. Read ZIP-day rows.
2. `slope_3d = (cases_today - cases_2_days_ago) / 2`.
3. Keep the last three such slopes (`lag0`, `lag1`, `lag2`) and the mean of the last 8 overlapping slopes (`avg_slope_10d`).
4. Look 7 days ahead for `cases_tplus7`, compute `r`, map to a bin, force `flat` when base &lt; 10.
5. Drop rows missing predictors or the future day.

Printed output to check: row counts, base≥10 share, label distribution, a few example rows.

In [2]:
def assign_change_label(cases_today: float, cases_tplus7: float, base_min: int = BASE_MIN) -> str:
    if pd.isna(cases_today) or pd.isna(cases_tplus7):
        return np.nan
    if cases_today < base_min:
        return "flat"
    r = (cases_tplus7 - cases_today) / cases_today
    if r <= -3.0:
        return "down_300p"
    if r <= -1.0:
        return "down_100p"
    if r <= -0.5:
        return "down_50p"
    if r < -0.1:
        return "down_10p"
    if r <= 0.1:
        return "flat"
    if r <= 0.5:
        return "up_50p"
    if r <= 1.0:
        return "up_100p"
    if r <= 3.0:
        return "up_300p"
    return "up_300p_plus"


# Tiny unit checks (must print flat for tiny base even if % is huge)
demo = pd.DataFrame([
    {"note": "tiny base 2->6 (+200%)", "cases_today": 2, "cases_tplus7": 6},
    {"note": "base 12->24 (+100%)", "cases_today": 12, "cases_tplus7": 24},
    {"note": "base 20->21 (+5%)", "cases_today": 20, "cases_tplus7": 21},
    {"note": "base 15->5 (-67%)", "cases_today": 15, "cases_tplus7": 5},
])
demo["label"] = [
    assign_change_label(a, b) for a, b in zip(demo.cases_today, demo.cases_tplus7)
]
print("Rule sanity checks")
display(demo)

need = [
    "zip", "date", "cases_today",
    "case_velocity", "case_acceleration", "case_jerk",
    "population", "population_density_per_sq_km",
]
panel = pd.read_csv(DATA, dtype={"zip": "string"}, parse_dates=["date"], usecols=need)
panel = panel.sort_values(["zip", "date"]).reset_index(drop=True)

g = panel.groupby("zip", sort=False)["cases_today"]
panel["slope_3d"] = (g.shift(0) - g.shift(2)) / 2.0
panel["slope_3d_lag1"] = panel.groupby("zip", sort=False)["slope_3d"].shift(1)
panel["slope_3d_lag2"] = panel.groupby("zip", sort=False)["slope_3d"].shift(2)
panel["avg_slope_10d"] = panel.groupby("zip", sort=False)["slope_3d"].transform(
    lambda s: s.rolling(8, min_periods=8).mean()
)
panel["cases_tplus7"] = g.shift(-HORIZON)
panel["rel_change_7d"] = np.where(
    panel["cases_today"] > 0,
    (panel["cases_tplus7"] - panel["cases_today"]) / panel["cases_today"],
    np.nan,
)
panel["label"] = [
    assign_change_label(a, b) for a, b in zip(panel.cases_today, panel.cases_tplus7)
]

FEATURE_COLUMNS = LABEL_RULES["predictors"]
model_df = panel.dropna(subset=FEATURE_COLUMNS + ["label", "cases_tplus7"]).copy()
model_df["label"] = pd.Categorical(model_df["label"], categories=LABEL_ORDER, ordered=True)

print("\nPanel rows:", f"{len(panel):,}")
print("Usable modeling rows:", f"{len(model_df):,}")
print("ZIPs:", model_df.zip.nunique())
print("Date range:", model_df.date.min().date(), "to", model_df.date.max().date())
print("Share of modeling rows with base >= 10:", f"{(model_df.cases_today >= BASE_MIN).mean():.3%}")

label_counts = model_df["label"].value_counts(dropna=False).reindex(LABEL_ORDER).fillna(0).astype(int)
label_share = (label_counts / label_counts.sum()).rename("share")
label_table = pd.concat([label_counts.rename("count"), label_share], axis=1)
label_table.to_csv(OUTPUT / "label_distribution.csv")
print("\nLabel distribution (after softening)")
display(label_table.style.format({"share": "{:.2%}"}))

print("\nExample rows (mixed bases)")
display(
    model_df.loc[:, ["zip", "date", "cases_today", "cases_tplus7", "rel_change_7d", "label"] + FEATURE_COLUMNS]
    .head(8)
)

Rule sanity checks


,note,cases_today,cases_tplus7,label
0,tiny base 2->6 (+200%),2,6,flat
1,base 12->24 (+100%),12,24,up_100p
2,base 20->21 (+5%),20,21,flat
3,base 15->5 (-67%),15,5,down_50p



Panel rows: 629,527
Usable modeling rows: 622,231
ZIPs: 456
Date range: 2020-02-24 to 2023-12-23
Share of modeling rows with base >= 10: 1.882%

Label distribution (after softening)


,count,share
label,,
down_300p,0,0.00%
down_100p,89,0.01%
down_50p,3188,0.51%
down_10p,4268,0.69%
flat,612371,98.42%
up_50p,1509,0.24%
up_100p,604,0.10%
up_300p,198,0.03%
up_300p_plus,4,0.00%



Example rows (mixed bases)


,zip,date,cases_today,cases_tplus7,rel_change_7d,label,case_velocity,case_acceleration,case_jerk,slope_3d,slope_3d_lag1,slope_3d_lag2,avg_slope_10d,population,population_density_per_sq_km
9,07002,2020-03-12,0,1.0,NaN,flat,0.0,0.0,0.0,0.0,0.0,0.0,-0.0625,70553,4678.097271
10,07002,2020-03-13,0,1.0,NaN,flat,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,70553,4678.097271
11,07002,2020-03-14,3,2.0,-0.333333,flat,3.0,3.0,3.0,1.5,0.0,0.0,0.1875,70553,4678.097271
12,07002,2020-03-15,2,3.0,0.500000,flat,-1.0,-4.0,-7.0,1.0,1.5,0.0,0.3125,70553,4678.097271
13,07002,2020-03-16,2,3.0,0.500000,flat,0.0,1.0,5.0,-0.5,1.0,1.5,0.2500,70553,4678.097271
14,07002,2020-03-17,1,1.0,0.000000,flat,-1.0,-1.0,-2.0,-0.5,-0.5,1.0,0.1875,70553,4678.097271
15,07002,2020-03-18,0,4.0,NaN,flat,-1.0,0.0,1.0,-1.0,-0.5,-0.5,0.0625,70553,4678.097271
16,07002,2020-03-19,1,1.0,0.000000,flat,1.0,2.0,2.0,0.0,-1.0,-0.5,0.0625,70553,4678.097271


## Phase 2 — Design matrix, chronological split, fit logistic

### Cell goal
Cut the calendar about 2/3 for training and 1/3 for testing (no shuffling — future days must not train the past). Fit a multinomial logistic regression with balanced class weights so rare up/down bins are not ignored.

### Abstract example
If dates run Jan→Dec, train might be Jan–Aug and test Sep–Dec. The model learns, from slopes/velocity/population, which change bin is more likely 7 days ahead.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

dates = pd.DatetimeIndex(sorted(model_df.date.unique()))
cutoff = dates[len(dates) * 2 // 3]
train = model_df.loc[model_df.date < cutoff].copy()
test = model_df.loc[model_df.date >= cutoff].copy()

print("Cutoff (test starts):", cutoff.date())
print(f"Train: {len(train):,} rows | {train.date.min().date()} to {train.date.max().date()}")
print(f"Test:  {len(test):,} rows | {test.date.min().date()} to {test.date.max().date()}")
print("\nTrain label mix")
display(train.label.value_counts().reindex(LABEL_ORDER).fillna(0).astype(int).to_frame("train_count"))
print("Test label mix")
display(test.label.value_counts().reindex(LABEL_ORDER).fillna(0).astype(int).to_frame("test_count"))

X_train = train[FEATURE_COLUMNS]
y_train = train["label"].astype(str)
X_test = test[FEATURE_COLUMNS]
y_test = test["label"].astype(str)

# Drop classes absent in train so logistic can fit; remap unused test labels if needed.
train_classes = sorted(y_train.unique(), key=lambda c: LABEL_ORDER.index(c))
print("Classes present in train:", train_classes)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        solver="lbfgs",
    )),
])
pipe.fit(X_train, y_train)
print("\nFitted multinomial logistic regression with StandardScaler + class_weight=balanced.")
print("Coefficients shape (classes x features):", pipe.named_steps["clf"].coef_.shape)

Cutoff (test starts): 2022-09-13
Train: 409,279 rows | 2020-02-24 to 2022-09-12
Test:  212,952 rows | 2022-09-13 to 2023-12-23

Train label mix


,train_count
label,
down_300p,0
down_100p,86
down_50p,3132
down_10p,4242
flat,399504
up_50p,1509
up_100p,604
up_300p,198
up_300p_plus,4


Test label mix


,test_count
label,
down_300p,0
down_100p,3
down_50p,56
down_10p,26
flat,212867
up_50p,0
up_100p,0
up_300p,0
up_300p_plus,0


Classes present in train: ['down_100p', 'down_50p', 'down_10p', 'flat', 'up_50p', 'up_100p', 'up_300p', 'up_300p_plus']

Fitted multinomial logistic regression with StandardScaler + class_weight=balanced.
Coefficients shape (classes x features): (8, 9)


## Phase 3 — Evaluate

### Cell goal
Score the logistic model on the held-out test days. Compare to a dumb baseline that always predicts `flat` (because softening + quiet days make flat common).

Look at:
- accuracy and macro-F1
- confusion matrix
- a few ZIP-day examples with actual vs predicted labels

### How to read macro-F1
Accuracy can look fine if the model mostly says `flat`. Macro-F1 averages performance across bins, so rare `up_300p_plus` mistakes still matter.

In [4]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

pred = pipe.predict(X_test)
baseline = np.full(len(y_test), "flat")

metrics = pd.DataFrame([
    {
        "model": "Logistic (balanced)",
        "accuracy": accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro", labels=train_classes, zero_division=0),
    },
    {
        "model": "Baseline: always flat",
        "accuracy": accuracy_score(y_test, baseline),
        "macro_f1": f1_score(y_test, baseline, average="macro", labels=train_classes, zero_division=0),
    },
])
metrics.to_csv(OUTPUT / "metrics_test.csv", index=False)
print("Test metrics")
display(metrics.round(4))

print("\nClassification report (logistic)")
print(classification_report(y_test, pred, labels=train_classes, zero_division=0))

cm = confusion_matrix(y_test, pred, labels=train_classes)
cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in train_classes], columns=[f"pred_{c}" for c in train_classes])
cm_df.to_csv(OUTPUT / "confusion_matrix_test.csv")
print("Confusion matrix (rows = true, columns = predicted)")
display(cm_df)

examples = test.loc[:, ["zip", "date", "cases_today", "cases_tplus7", "rel_change_7d", "label"]].copy()
examples["predicted"] = pred
examples["baseline"] = "flat"
# show some non-flat truths if any, else head
interesting = examples.loc[examples.label.astype(str).ne("flat")].head(8)
if interesting.empty:
    interesting = examples.head(8)
print("\nExample test rows")
display(interesting)

Test metrics


,model,accuracy,macro_f1
0,Logistic (balanced),0.8514,0.1176
1,Baseline: always flat,0.9996,0.1250



Classification report (logistic)
              precision    recall  f1-score   support

   down_100p       0.00      0.33      0.00         3
    down_50p       0.01      0.39      0.02        56
    down_10p       0.00      0.19      0.00        26
        flat       1.00      0.85      0.92    212867
      up_50p       0.00      0.00      0.00         0
     up_100p       0.00      0.00      0.00         0
     up_300p       0.00      0.00      0.00         0
up_300p_plus       0.00      0.00      0.00         0

    accuracy                           0.85    212952
   macro avg       0.13      0.22      0.12    212952
weighted avg       1.00      0.85      0.92    212952

Confusion matrix (rows = true, columns = predicted)


,pred_down_100p,pred_down_50p,pred_down_10p,pred_flat,pred_up_50p,pred_up_100p,pred_up_300p,pred_up_300p_plus
true_down_100p,1,0,1,0,0,0,0,1
true_down_50p,9,22,7,0,0,3,1,14
true_down_10p,2,5,5,0,1,3,2,8
true_flat,3359,2265,6201,181284,12199,3445,1604,2510
true_up_50p,0,0,0,0,0,0,0,0
true_up_100p,0,0,0,0,0,0,0,0
true_up_300p,0,0,0,0,0,0,0,0
true_up_300p_plus,0,0,0,0,0,0,0,0



Example test rows


,zip,date,cases_today,cases_tplus7,rel_change_7d,label,predicted,baseline
2397,07003,2022-11-27,11,4.0,-0.636364,down_50p,down_50p,flat
34471,07032,2022-09-19,11,3.0,-0.727273,down_50p,down_100p,flat
34541,07032,2022-11-28,10,5.0,-0.500000,down_50p,down_10p,flat
34555,07032,2022-12-12,10,4.0,-0.600000,down_50p,down_50p,flat
34590,07032,2023-01-16,12,3.0,-0.750000,down_50p,down_50p,flat
37366,07035,2023-01-03,11,1.0,-0.909091,down_50p,down_50p,flat
37394,07035,2023-01-31,13,0.0,-1.000000,down_100p,down_100p,flat
42945,07042,2022-12-05,11,5.0,-0.545455,down_50p,down_100p,flat


## Phase 4 — Save predictions and takeaway

Writes the full test prediction table and a short coefficient peek (which features push which classes).

In [5]:
pred_out = test.loc[:, ["zip", "date", "cases_today", "cases_tplus7", "rel_change_7d", "label"]].copy()
pred_out["predicted"] = pred
pred_out.to_csv(OUTPUT / "predictions_test.csv", index=False)

clf = pipe.named_steps["clf"]
coef = pd.DataFrame(clf.coef_, index=clf.classes_, columns=FEATURE_COLUMNS)
coef.to_csv(OUTPUT / "logistic_coefficients.csv")
print("Saved predictions_test.csv and logistic_coefficients.csv")
print("\nCoefficient peek (rows = classes, columns = features)")
display(coef.round(3))

print("\nFiles in", OUTPUT.resolve())
for path in sorted(OUTPUT.iterdir()):
    print(" -", path.name)

Saved predictions_test.csv and logistic_coefficients.csv

Coefficient peek (rows = classes, columns = features)


,case_velocity,case_acceleration,case_jerk,slope_3d,slope_3d_lag1,slope_3d_lag2,avg_slope_10d,population,population_density_per_sq_km
down_100p,0.109,0.037,0.049,0.148,0.173,-0.039,-0.072,-0.217,-0.065
down_10p,0.067,0.018,0.001,0.100,0.046,0.024,-0.074,0.155,0.014
down_50p,0.129,0.033,-0.002,0.195,0.080,0.037,-0.132,-0.131,0.029
flat,-0.153,-0.071,-0.025,-0.180,-0.021,-0.082,-0.241,-2.077,-0.227
up_100p,-0.041,-0.015,-0.010,-0.054,-0.035,-0.001,0.079,0.377,-0.003
up_300p,-0.073,-0.017,0.015,-0.114,-0.011,-0.050,0.156,0.358,0.120
up_300p_plus,-0.018,0.022,-0.034,-0.073,-0.248,0.126,0.260,1.229,0.107
up_50p,-0.018,-0.008,0.004,-0.023,0.016,-0.016,0.023,0.305,0.025



Files in /Users/dhruvesh/Desktop/Fall 2026/Analytics/Jupyter/Dhruvesh/change_bin_7d_outputs
 - confusion_matrix_test.csv
 - label_distribution.csv
 - label_rules.json
 - logistic_coefficients.csv
 - metrics_test.csv
 - predictions_test.csv


## Takeaway

- **Model job:** classify 7-day relative change bins, with base &lt; 10 forced to `flat`.
- **Predictors:** velocity / acceleration / jerk, last three 3-day slopes, 10-day average slope, population, density.
- **Softening is in the label rule**, not a custom loss.
- **Expect lots of `flat`:** many ZIP-days are quiet; judge success with macro-F1 and the confusion matrix, not accuracy alone.
- Rules live in `change_bin_7d_outputs/label_rules.json` so the team can reuse the same definition.